In [ ]:
!pip install transformers trl peft datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = 'Qwen/Qwen2.5-0.5B-Instruct'

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype = 'auto',
    device_map = 'auto'
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
from datasets import load_dataset
ds = load_dataset("HuggingFaceH4/ultrafeedback_binarized", revision="292c16329d921287c4166934cac1a6ad1e13a6c5")
print(ds)

README.md:   0%|          | 0.00/5.98k [00:00<?, ?B/s]

data/train_sft-00000-of-00001-49a5a3acc4(…):   0%|          | 0.00/222M [00:00<?, ?B/s]

data/test_sft-00000-of-00001-0813b122475(…):   0%|          | 0.00/3.50M [00:00<?, ?B/s]

data/train_gen-00000-of-00001-4f6b46acdf(…):   0%|          | 0.00/180M [00:00<?, ?B/s]

data/test_gen-00000-of-00001-bc700fffaa8(…):   0%|          | 0.00/2.84M [00:00<?, ?B/s]

...

Generating train_gen split:   0%|          | 0/61966 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train_prefs split:   0%|          | 0/61966 [00:00<?, ? examples/s]

Generating test_prefs split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train_sft: Dataset({
        features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected'],
        num_rows: 61966
    })
    test_sft: Dataset({
        features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected'],
        num_rows: 1000
    })
    train_gen: Dataset({
        features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected'],
        num_rows: 61966
    })
    test_gen: Dataset({
        features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected'],
        num_rows: 1000
    })
    train_prefs: Dataset({
        features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected'],
        num_rows: 61966
    })
    test_prefs: Dataset({
        features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected'],
        num_rows

In [ ]:
print(ds["train_sft"]["chosen"][0])

[{'content': 'Write a 1,000-word op-ed piece in a formal tone, analyzing and providing examples of the ways in which social media platforms have been utilized to spread extremist and violent ideologies. In your analysis, discuss the specific tactics that these groups use to spread their messages online and the effects of these tactics on both individuals and society. Additionally, provide possible solutions that could be implemented to combat the spread of these dangerous ideologies on social media. Your piece should be well-researched, citing reputable sources to support your arguments.', 'role': 'user'}, {'content': "Title: The Noxious Alliance of Social Media and Extremism: A Threat to Human Progress\n\nThe advent of social media platforms has irrevocably revolutionized the way we communicate, share and consume information. While these platforms provide access to information for a global audience and allow individuals to connect regardless of geographical barriers, they have also be

In [ ]:
def formatting_logs(data):
  # print(data["chosen"][0].keys())
  prompt = data["chosen"][0].get("content")
  completion = data["chosen"][1].get("content")
  return {
      "prompt": [{"role": "user", "content": f"{prompt}"}],
      "completion": [{"role": "assistant", "content": f"{completion}"}]
  }

train_data_sft = ds["train_sft"].map(formatting_logs).select_columns(["prompt", "completion"])
eval_data_sft = ds["test_sft"].map(formatting_logs).select_columns(["prompt", "completion"])

Map:   0%|          | 0/61966 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
print(train_data_sft[0])

{'prompt': [{'role': 'user', 'content': 'Write a 1,000-word op-ed piece in a formal tone, analyzing and providing examples of the ways in which social media platforms have been utilized to spread extremist and violent ideologies. In your analysis, discuss the specific tactics that these groups use to spread their messages online and the effects of these tactics on both individuals and society. Additionally, provide possible solutions that could be implemented to combat the spread of these dangerous ideologies on social media. Your piece should be well-researched, citing reputable sources to support your arguments.'}], 'completion': [{'role': 'assistant', 'content': "Title: The Noxious Alliance of Social Media and Extremism: A Threat to Human Progress\n\nThe advent of social media platforms has irrevocably revolutionized the way we communicate, share and consume information. While these platforms provide access to information for a global audience and allow individuals to connect regard

In [ ]:
!pip3 install trl[peft]

In [ ]:
!mkdir /content/sft

In [ ]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 57.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
train_data_sft_final = train_data_sft.select(range(5000))

In [ ]:
import torch, gc
from accelerate.state import AcceleratorState, PartialState

AcceleratorState._reset_state()
PartialState._reset_state()

gc.collect()
torch.cuda.empty_cache()

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

training_args = SFTConfig(
    output_dir='/content/sft',
    learning_rate=2.0e-4,
    num_train_epochs=1,
    logging_steps=10,
    max_length=512,
    bf16=True,
    save_strategy="no",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data_sft_final,
    eval_dataset=eval_data_sft,
    peft_config=peft_config,
)

trainer.train()


/tmp/ipykernel_1433/3372927351.py:15: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(
/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will le

Step,Training Loss
10,1.488896
20,1.358323
30,1.244778
40,1.306536
50,1.308080
60,1.268397
70,1.387053
80,1.209708
90,1.327322
100,1.316536


TrainOutput(global_step=625, training_loss=1.3209805953979492, metrics={'train_runtime': 3226.2919, 'train_samples_per_second': 1.55, 'train_steps_per_second': 0.194, 'total_flos': 3918545287008000.0, 'train_loss': 1.3209805953979492, 'entropy': 1.2725442722439766, 'mean_token_accuracy': 0.6260210789740086, 'num_tokens': 1813835.0, 'epoch': 1.0})

In [ ]:
trainer.save_model('/content/sft')
tokenizer.save_pretrained('/content/sft')

('/content/sft/tokenizer_config.json',
 '/content/sft/chat_template.jinja',
 '/content/sft/tokenizer.json')

In [ ]:
!zip -r sft.zip /content/sft/

  adding: content/sft/ (stored 0%)
  adding: content/sft/chat_template.jinja (deflated 71%)
  adding: content/sft/tokenizer_config.json (deflated 59%)
  adding: content/sft/README.md (deflated 65%)
  adding: content/sft/tokenizer.json (deflated 81%)
  adding: content/sft/training_args.bin (deflated 53%)
  adding: content/sft/adapter_config.json (deflated 59%)
  adding: content/sft/adapter_model.safetensors (deflated 7%)
